In [0]:
%run ../Includes/Copy-Datasets

In [0]:
 %sql
 ALTER TABLE orders_silver ADD CONSTRAINT timestamp_within_range CHECK (order_timestamp >= '2020-01-01');

In [0]:
 %sql
 DESCRIBE EXTENDED orders_silver

In [0]:
 %sql
 INSERT INTO orders_silver
 VALUES ('1', '2022-02-01 00:00:00.000', 'C00001', 0, 0, NULL),
        ('2', '2019-05-01 00:00:00.000', 'C00001', 0, 0, NULL),
        ('3', '2023-01-01 00:00:00.000', 'C00001', 0, 0, NULL)

In [0]:
 %sql
 SELECT *
 FROM orders_silver
 WHERE order_id IN ('1', '2', '3')

In [0]:
 %sql
 ALTER TABLE orders_silver ADD CONSTRAINT valid_quantity CHECK (quantity > 0);

In [0]:
%sql
 DESCRIBE EXTENDED orders_silver

In [0]:
 %sql
 SELECT *
 FROM orders_silver
 where quantity <= 0

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

query = (spark.readStream.table("bronze")
        .filter("topic = 'orders'")
        .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
        .select("v.*")
        .filter("quantity > 0")
     .writeStream
        .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/orders_silver")
        .trigger(availableNow=True)
        .table("orders_silver"))

query.awaitTermination()

In [0]:
 %sql
 ALTER TABLE orders_silver DROP CONSTRAINT timestamp_within_range;


In [0]:
 %sql
 DESCRIBE EXTENDED orders_silver


In [0]:
%sql
 DROP TABLE orders_silver

In [0]:
dbutils.fs.rm("dbfs:/mnt/demo_pro/checkpoints/orders_silver", True)

